In [0]:
import pandas as pd
import requests
from datetime import datetime

# URLs já conferidas manualmente —
urls_por_ano = {
    2015: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2015_csv.zip",
    2016: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2016_csv.zip",
    2017: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2017_csv.zip",
    2018: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2018_csv.zip",
    2019: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2019_csv.zip",
    2020: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2020_csv.zip",
    2021: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_2021_csv.zip",
    2022: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/DO22OPEN.csv",
    2023: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/DO23OPEN.csv",
    2024: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/csv/DO24OPEN_csv.zip",
}

tabela_bronze = "bronze_mortalidade_sim"


def descobrir_url(ano):
    """Pra anos ainda não mapeados manualmente, tenta os padrões de nome
    já vistos no portal do SIM. Retorna a 1ª URL que responder OK, ou
    None se nenhuma funcionar (aí precisa checar na mão)."""
    aa = str(ano)[-2:]
    candidatos = [
        f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/DO{aa}OPEN.csv",
        f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/Mortalidade_Geral_{ano}_csv.zip",
        f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/csv/DO{aa}OPEN_csv.zip",
    ]
    for url in candidatos:
        try:
            resp = requests.head(url, timeout=15)
            if resp.status_code == 200:
                return url
        except requests.RequestException:
            continue
    return None


# 1) Quais anos já estão gravados na tabela?
try:
    anos_existentes = {
        row["ANO_REFERENCIA"]
        for row in spark.sql(f"SELECT DISTINCT ANO_REFERENCIA FROM {tabela_bronze}").collect()
    }
except Exception:
    anos_existentes = set()  # tabela ainda não existe: é a primeira execução

print("Anos já gravados:", sorted(anos_existentes) if anos_existentes else "nenhum ainda")

# 2) Além dos anos mapeados, tenta descobrir anos novos (do próximo ano
#    após o último mapeado até o ano anterior ao atual — não mexe no ano
#    corrente porque ele costuma estar incompleto)
ano_atual = datetime.now().year
ultimo_ano_mapeado = max(urls_por_ano.keys())
anos_para_processar = dict(urls_por_ano)

for ano in range(ultimo_ano_mapeado + 1, ano_atual):
    if str(ano) in anos_existentes:
        continue
    url_descoberta = descobrir_url(ano)
    if url_descoberta:
        anos_para_processar[ano] = url_descoberta
        print(f"Ano {ano}: URL nova descoberta automaticamente -> {url_descoberta}")
    else:
        print(f"Ano {ano}: nenhum padrão conhecido funcionou — checar manualmente.")

# 3) Processa só o que realmente falta
for ano, url in sorted(anos_para_processar.items()):
    if str(ano) in anos_existentes:
        print(f"Ano {ano} já está gravado, pulando.")
        continue

    print(f"Baixando {ano}...")
    df_ano = pd.read_csv(url, sep=";", encoding="ISO-8859-1", dtype=str, low_memory=False)
    df_ano["ANO_REFERENCIA"] = str(ano)

    spark_df = spark.createDataFrame(df_ano)
    (spark_df.write
        .format("delta")
        .mode("append")           # sempre append: cria a tabela se não existir
        .option("mergeSchema", "true")
        .saveAsTable(tabela_bronze))

    print(f"Ano {ano}: {len(df_ano)} linhas gravadas em '{tabela_bronze}'")

print("Ingestão bronze concluída (incremental).")

In [0]:
tabela_bronze = "bronze_mortalidade_sim"

display(spark.sql(f"SELECT ANO_REFERENCIA, COUNT(*) as qtd FROM {tabela_bronze} GROUP BY ANO_REFERENCIA ORDER BY ANO_REFERENCIA"))